In [9]:
import altair as alt
import pandas as pd
import numpy as np
from vega_datasets import data
from datetime import datetime

In [2]:
df = pd.read_csv("all_parameters_daily.csv")

C:\Users\jmei7\AppData\Local\Temp\ipykernel_34740\929220242.py:1: DtypeWarning: Columns (13,27) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("all_parameters_daily.csv")


In [4]:
df.columns[[13, 27]]

Index(['Event Type', 'CBSA Name'], dtype='object')

In [12]:
subset = df[['Date Local', 'AQI', 'Latitude', 'Longitude', 'Local Site Name', 'CBSA Name']]
subset.head()

,Date Local,AQI,Latitude,Longitude,Local Site Name,CBSA Name
0,2025-06-16,51.0,41.995568,-83.946559,"6792 RAISIN CENTER HWY, LENAWEE CO.RD.COMM.OWN...","Adrian, MI"
1,2025-06-15,74.0,41.995568,-83.946559,"6792 RAISIN CENTER HWY, LENAWEE CO.RD.COMM.OWN...","Adrian, MI"
2,2025-06-14,40.0,41.995568,-83.946559,"6792 RAISIN CENTER HWY, LENAWEE CO.RD.COMM.OWN...","Adrian, MI"
3,2025-06-13,32.0,41.995568,-83.946559,"6792 RAISIN CENTER HWY, LENAWEE CO.RD.COMM.OWN...","Adrian, MI"
4,2025-06-12,47.0,41.995568,-83.946559,"6792 RAISIN CENTER HWY, LENAWEE CO.RD.COMM.OWN...","Adrian, MI"


In [56]:
subset['AQI'].median()

28.0

In [123]:
locations = subset[['Latitude', 'Longitude', 'Local Site Name', 'CBSA Name']].drop_duplicates("Local Site Name").dropna()

In [124]:
locations

,Latitude,Longitude,Local Site Name,CBSA Name
0,41.995568,-83.946559,"6792 RAISIN CENTER HWY, LENAWEE CO.RD.COMM.OWN...","Adrian, MI"
215,41.182466,-81.330486,Lake Rockwell,"Akron, OH"
223,41.063526,-81.468956,East HS,"Akron, OH"
838,41.103028,-81.496253,North HS,"Akron, OH"
1297,43.012090,-73.648900,STILLWATER,"Albany-Schenectady-Troy, NY"
...,...,...,...,...
550888,41.427100,-80.145100,M.K. Goddard,"Youngstown-Warren-Boardman, OH-PA"
551196,41.215014,-80.484779,Farrell,"Youngstown-Warren-Boardman, OH-PA"
552119,39.205572,-121.820362,Sutter Buttes (seasonal),"Yuba City, CA"
552158,39.138773,-121.618549,Yuba City,"Yuba City, CA"


In [42]:
datetime.today().timetuple().tm_yday

97

### PARAMS

In [173]:
# select subsection geographically
brush = alt.selection_interval(name="zoom")

# parameter choices
choice_a = alt.binding_select(options = ['Ozone', 'SO2', 'NO2', 'CO'],
                                     name = "Parameter Name: " )
choice_b = alt.binding_select(options = ['Ozone', 'SO2', 'NO2', 'CO'],
                                     name = "Parameter Name: " )
# time selection

day_slider = alt.binding_range(min=1, max=365, step=1, name="Day: ")
day_select = alt.param(name="selected_day", bind=day_slider, value=5)

# site select
site_selection = alt.selection_point(fields=['Local Site Name'], on = 'mouseover', empty=False)

# selections
selection_a = alt.selection_point(fields = ['Parameter Name'], bind = choice_a)
selection_b = alt.selection_point(fields = ['Parameter Name'], bind = choice_b)

parameters = df[["Date Local", "Parameter Name", "Arithmetic Mean", "AQI", "Local Site Name", "Longitude", "Latitude"]]
parameters["Day"] = [datetime.strptime(i, "%Y-%m-%d").timetuple().tm_yday for i in parameters["Date Local"]]


C:\Users\jmei7\AppData\Local\Temp\ipykernel_34740\1251476301.py:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  parameters["Day"] = [datetime.strptime(i, "%Y-%m-%d").timetuple().tm_yday for i in parameters["Date Local"]]


### MAIN MAP

In [176]:
# geomap

usa = data.us_10m.url
geo = alt.layer(

    alt.Chart(alt.topo_feature(usa, 'states')).mark_geoshape(
        fill = '#e6f3f7', stroke = '#000', strokeWidth = 1
        ).project(
            type = 'albersUsa'
        ).properties(
            width = 800, height = 500
        ),

    alt.Chart(locations).mark_circle(size = 8).encode(
        longitude='Longitude:Q',
        latitude='Latitude:Q',
        tooltip=['Local Site Name:N', 'CBSA Name:N'],
        color=alt.condition(site_selection, alt.value('orange'), alt.value('grey'))
    )

).project(
    type = 'albersUsa'
).properties(
    width = 800, height = 500
).add_params(site_selection)


### FACET

In [74]:
# facet
# on hover: take points and compare pollutant levels at that point in time (bar)
# might not have both
# default to us median



C:\Users\jmei7\AppData\Local\Temp\ipykernel_34740\1932230803.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  parameters["Day"] = [datetime.strptime(i, "%Y-%m-%d").timetuple().tm_yday for i in parameters["Date Local"]]


,Date Local,Parameter Name,Arithmetic Mean,AQI,Local Site Name,Longitude,Latitude,Day
0,2025-06-16,Ozone,0.034800,51.0,"6792 RAISIN CENTER HWY, LENAWEE CO.RD.COMM.OWN...",-83.946559,41.995568,167
1,2025-06-15,Ozone,0.046059,74.0,"6792 RAISIN CENTER HWY, LENAWEE CO.RD.COMM.OWN...",-83.946559,41.995568,166
2,2025-06-14,Ozone,0.030824,40.0,"6792 RAISIN CENTER HWY, LENAWEE CO.RD.COMM.OWN...",-83.946559,41.995568,165
3,2025-06-13,Ozone,0.031882,32.0,"6792 RAISIN CENTER HWY, LENAWEE CO.RD.COMM.OWN...",-83.946559,41.995568,164
4,2025-06-12,Ozone,0.040176,47.0,"6792 RAISIN CENTER HWY, LENAWEE CO.RD.COMM.OWN...",-83.946559,41.995568,163


In [102]:
parameters[parameters["Day"]==5]

,Date Local,Parameter Name,Arithmetic Mean,AQI,Local Site Name,Longitude,Latitude,Day
756,2025-01-05,Sulfur dioxide,0.000000,NaN,East HS,-81.468956,41.063526,5
850,2025-01-05,Sulfur dioxide,0.000000,0.0,East HS,-81.468956,41.063526,5
1633,2025-01-05,Ozone,0.031471,33.0,STILLWATER,-73.648900,43.012090,5
1827,2025-01-05,Ozone,0.034706,36.0,2ZJ Bernalillo,-106.548914,35.299484,5
2153,2025-01-05,Ozone,0.033118,39.0,2LL Los Lunas,-106.739600,34.814700,5
...,...,...,...,...,...,...,...,...
597412,2025-01-05,Sulfur dioxide,1.000000,NaN,Town of Sinclair SO2 Monitor,-107.120830,41.782370,5
597583,2025-01-05,Sulfur dioxide,0.333333,1.0,Sinclair North East (Ambient),-107.084220,41.793580,5
597764,2025-01-05,Sulfur dioxide,0.312500,NaN,Sinclair North East (Ambient),-107.084220,41.793580,5
597945,2025-01-05,Sulfur dioxide,0.587500,0.0,Wyoming Refining,-104.205120,43.845390,5


In [101]:
parameters.dtypes

Date Local          object
Parameter Name      object
Arithmetic Mean    float64
AQI                float64
Local Site Name     object
Longitude          float64
Latitude           float64
Day                  int64
dtype: object

In [179]:
alt.data_transformers.disable_max_rows()
param_bar = alt.Chart(parameters).mark_bar().encode(
    x=alt.X("Parameter Name:N"),
    y=alt.Y("Arithmetic Mean:Q")
).add_params(
    day_select
).transform_filter(
    alt.datum.Day == day_select
).transform_filter(
    site_selection
)

In [ ]:
geo | param_bar